# Product Review Summarizer

Turn a collection of customer reviews into a concise buying guide using an LLM.

This beginner-friendly Week 1 project demonstrates:

- loading an OpenAI API key from a `.env` file
- designing system and user prompts
- sending messages to the Chat Completions API
- streaming and displaying a Markdown response

> The result is an AI-generated summary, not a substitute for reading the original reviews.

## 1. Imports and setup

Place your OpenAI API key in the project `.env` file as `OPENAI_API_KEY=...`.

In [ ]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found. Add it to your .env file and try again.")

openai = OpenAI(api_key=api_key)
print("OpenAI client is ready.")

## 2. Sample product reviews

The example uses fictional reviews, so the notebook works without scraping a website. Replace these with reviews for any product you want to analyze.

In [ ]:
product_name = "SoundWave Mini Wireless Speaker"

reviews = [
    {"rating": 5, "text": "Surprisingly clear sound for such a small speaker. Pairing was quick and the battery lasted through two workdays."},
    {"rating": 4, "text": "Good value and easy to carry. Bass is fine indoors, but it gets weak at high volume."},
    {"rating": 2, "text": "The Bluetooth connection dropped twice when my phone was in the next room. The buttons are also hard to see."},
    {"rating": 5, "text": "I use it in the kitchen every day. It sounds clear, connects instantly, and charges with USB-C."},
    {"rating": 3, "text": "Decent speaker for podcasts, but music sounds thin outdoors. Battery life was about ten hours for me."},
    {"rating": 4, "text": "Compact and well built. The volume controls could be more responsive, but I would buy it again for travel."},
]

print(f"Loaded {len(reviews)} reviews for {product_name}.")

## 3. Build the prompts

The system prompt defines the assistant's role and output format. The user prompt supplies the product name and numbered reviews.

In [ ]:
system_prompt = """You are a careful product-review analyst.
Analyze only the reviews supplied by the user. Do not invent product features or customer opinions.
Distinguish repeated feedback from isolated comments and mention when the sample is small.

Respond in Markdown using exactly these sections:
# Review Summary
## Overall Sentiment
## What Customers Like
## Common Complaints
## Best For
## Think Twice If
## Verdict

In the verdict, choose one label: Recommended, Mixed, or Not Recommended, and explain it briefly.
Do not wrap the response in a code block."""

In [ ]:
def user_prompt_for(product, product_reviews):
    if not product.strip():
        raise ValueError("Please provide a product name.")
    if not product_reviews:
        raise ValueError("Please provide at least one review.")

    formatted_reviews = []
    for number, review in enumerate(product_reviews, start=1):
        rating = review.get("rating", "not provided")
        text = review.get("text", "").strip()
        if text:
            formatted_reviews.append(f"Review {number} ({rating}/5 stars): {text}")

    if not formatted_reviews:
        raise ValueError("The reviews do not contain any text.")

    reviews_text = "\n".join(formatted_reviews)
    return f"""Analyze the following reviews for {product}.

Number of usable reviews: {len(formatted_reviews)}

{reviews_text}"""


def messages_for(product, product_reviews):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(product, product_reviews)},
    ]

It is useful to inspect the prompt before spending an API call.

In [ ]:
print(user_prompt_for(product_name, reviews))

## 4. Generate the review summary

This function streams the response and returns the complete Markdown result.

In [ ]:
def summarize_reviews(product, product_reviews, model="gpt-5-nano"):
    stream = openai.chat.completions.create(
        model=model,
        messages=messages_for(product, product_reviews),
        stream=True,
    )

    result = ""
    for chunk in stream:
        text = chunk.choices[0].delta.content or ""
        result += text

    return result

In [ ]:
summary = summarize_reviews(product_name, reviews)
display(Markdown(summary))

## 5. Try your own product

Edit the product name and reviews below, then run the cell. Ratings are optional; use `"not provided"` when you only have review text.

In [ ]:
my_product = "Your product name"
my_reviews = [
    {"rating": 5, "text": "Paste the first customer review here."},
    {"rating": 3, "text": "Paste the second customer review here."},
    {"rating": 1, "text": "Paste the third customer review here."},
]

# Uncomment these lines after replacing the example text above.
# my_summary = summarize_reviews(my_product, my_reviews)
# display(Markdown(my_summary))

## Ideas to extend the project

- Compare summaries from OpenAI and a local Ollama model.
- Group feedback by product feature such as battery, comfort, or price.
- Add a second product and ask the model for a side-by-side comparison.
- Read reviews from a CSV file instead of entering them in Python.